# Filtering & Categorical Encoding in a Case Study

01 Core Python · 02 Pandas · 03 Cleaning · 04 Transformation · **▶ 05 Feature Engineering** · 06 Regression · 07 Model Prep · 08 Case Studies

`05_Feature_Engineering/02_filtering_and_categorical_encoding.ipynb`

---

### In one paragraph (no jargon)

Feature engineering and filtering in the middle of a real analysis, where the decisions interact: which rows to exclude, which categories to collapse, and how to encode what remains. The banking dataset here has the high-cardinality columns that make encoding a real choice rather than a formality.

### After this notebook you can

- Filter rows on business rules and document what was excluded
- Collapse rare categories before encoding to keep the width manageable
- Apply the encoding chosen in section 04 to a genuine case study


### What's inside

1. Setup
2. Filtering rows
3. Handling rare categories
4. Encoding
5. Checking the result
6. Exam quick-reference

---

> **▶ Runs on its own.** The next cell is the only setup you need. It imports the
> libraries and loads the data. If the `datasets/` folder isn't where it expects,
> it rebuilds an equivalent dataset in memory so **every cell below still runs** —
> handy if you copy this single `.ipynb` somewhere else.
>
> **▶ Reading the cells.** Code comments explain *what the line does*; the text
> blocks explain *why you'd do it*. Look for these markers:
> `# WHAT:` a plain-English translation · `# WHY:` the reason it matters ·
> `# 🔧 CHANGE THIS:` the knob to turn when the exam question differs ·
> **⚡ Beyond the syllabus** = optional, higher-mark techniques.

### Jargon buster

| Term | Plain English |
|---|---|
| **Feature** | One column used as an input to a model. |
| **Feature engineering** | Creating better columns from the ones you have — the step that usually improves a model more than changing the algorithm. |
| **Feature extraction** | Pulling a new column out of an existing one: month out of a date, title out of a name. |
| **Feature splitting** | Breaking one column into several: `"Yangon(Local)"` → city + type. |
| **Derived feature** | A column computed from others: profit = revenue − cost. |
| **Domain knowledge** | Knowing what the numbers mean in the business. This is what makes a good feature; no algorithm can supply it. |

In [1]:
# =============================================================================
# SETUP — run this cell first. It is the only cell with dependencies.
# =============================================================================
# WHAT: `import` pulls in code other people have written so we don't rewrite it.
#       The `as pd` part is a nickname, so we can type `pd` instead of `pandas`.
import pandas as pd          # tables of data (think: Excel, but programmable)
import numpy as np           # fast maths on whole columns at once
import matplotlib.pyplot as plt   # charts
import warnings

warnings.filterwarnings('ignore')          # hide version-upgrade notices, keeps output readable
pd.set_option('display.max_columns', 50)   # don't hide columns behind "..."
pd.set_option('display.width', 160)
import os
import seaborn as sns
sns.set_theme(style="whitegrid")

def find_datasets_folder(start=None):
    """Walk upwards from this notebook looking for the shared `datasets/` folder.

    WHY: it means the notebook works whether you opened it from its own folder,
    from the top of the notes, or from anywhere else on your machine.
    """
    here = os.path.abspath(start or os.getcwd())
    for _ in range(6):                       # look up to 6 folders up
        candidate = os.path.join(here, 'datasets')
        if os.path.isdir(candidate):
            return candidate
        parent = os.path.dirname(here)
        if parent == here:
            break
        here = parent
    return None

def dataset_path(filename, rebuild=None):
    """Return a real path to `filename`, materialising a temp copy if it's missing.

    WHY: a few pandas tools (pd.ExcelFile, pd.read_sql) need an actual file path
         rather than a DataFrame, so the fallback has to be written to disk.
    """
    folder = find_datasets_folder()
    if folder:
        path = os.path.join(folder, filename)
        if os.path.exists(path):
            return path
    if rebuild is None:
        raise FileNotFoundError(filename)
    import tempfile
    tmp = os.path.join(tempfile.mkdtemp(prefix='bda_'), filename)
    frame = rebuild()
    (frame.to_excel(tmp, index=False) if filename.lower().endswith(('.xlsx', '.xls'))
     else frame.to_csv(tmp, index=False))
    print(f"'{filename}' not found -> wrote a rebuilt copy to {tmp}")
    return tmp

def load_data(filename, rebuild=None, **read_kwargs):
    """Load `filename` from the shared datasets folder, or rebuild it in memory.

    WHAT: tries to read the real file; if it can't find it, calls `rebuild()`
          which recreates a dataset with the same columns and behaviour.
    WHY:  guarantees this notebook runs even if the CSV goes missing.
    """
    folder = find_datasets_folder()
    if folder:
        path = os.path.join(folder, filename)
        if os.path.exists(path):
            reader = pd.read_excel if filename.lower().endswith(('.xlsx', '.xls')) else pd.read_csv
            print(f"Loaded '{filename}' from {folder}")
            return reader(path, **read_kwargs)
    if rebuild is None:
        raise FileNotFoundError(f"Could not find {filename} and no fallback was supplied.")
    print(f"'{filename}' not found on disk -> rebuilding an equivalent dataset in memory.")
    built = rebuild()
    if 'chunksize' in read_kwargs:            # keep chunked reads working on the fallback path
        size = read_kwargs['chunksize']
        return (built.iloc[i:i + size] for i in range(0, len(built), size))
    return built

def rebuild_banking():
    """Rebuild a dataset statistically equivalent to banking.csv (41,199 rows).

    Same columns, same types, same ranges and category mix, so every cell
    below still runs if the original file is missing."""
    rng = np.random.default_rng(42)
    n = 41199
    frame = pd.DataFrame({
        'age': rng.normal(40.02, 10.43, n).clip(1, 104).round().astype(int),
        'job': rng.choice(['admin.', 'blue-collar', 'technician', 'services', 'management', 'retired', 'entrepreneur', 'self-employed', 'housemaid', 'unemployed', 'student', 'unknown'], n, p=[0.253, 0.2246, 0.1637, 0.0964, 0.071, 0.0418, 0.0353, 0.0345, 0.0258, 0.0246, 0.0213, 0.008]),
        'marital': rng.choice(['married', 'single', 'divorced', 'unknown'], n, p=[0.6052, 0.2809, 0.1119, 0.002]),
        'education': rng.choice(['university.degree', 'high.school', 'basic.9y', 'professional.course', 'basic.4y', 'basic.6y', 'unknown', 'illiterate', 'Basic'], n, p=[0.2954, 0.2311, 0.1467, 0.1273, 0.1014, 0.0556, 0.042, 0.0004, 0.0001]),
        'default': rng.choice(['no', 'unknown', 'yes'], n, p=[0.7912, 0.2088, 0.0]),
        'housing': rng.choice(['yes', 'no', 'unknown'], n, p=[0.5238, 0.4522, 0.024]),
        'loan': rng.choice(['no', 'yes', 'unknown', 'n', 'y'], n, p=[0.8241, 0.1517, 0.024, 0.0001, 0.0001]),
    })
    return frame

def rebuild_customer():
    """Exact copy of customer.csv, embedded so this notebook never needs the file."""
    import io
    csv_text = """Cust_ID,Age,Income,Profession,Marital_Status,Vehicle_Type
C001,24,350000,Student,Single,Bike
C002,28,520000,Software Engineer,Married,Hatchback
C003,35,780000,Doctor,Married,SUV
C004,42,1250000,Business,Married,Sedan
C005,31,610000,Teacher,Single,Scooter
C006,45,980000,Banker,Married,Sedan
C007,29,430000,Sales Executive,Single,Bike
C008,38,890000,Professor,Married,SUV
C009,50,1450000,Business,Married,Luxury Car
C010,27,470000,Accountant,Single,Hatchback
C011,33,720000,Engineer,Married,Sedan
C012,41,860000,Government Employee,Married,SUV
C013,26,390000,Nurse,Single,Scooter
C014,36,810000,Lawyer,Married,Sedan
C015,48,1350000,Entrepreneur,Married,Luxury Car
C016,23,320000,Student,Single,Bike
C017,39,920000,IT Consultant,Married,SUV
C018,44,1120000,Business,Married,Sedan
C019,30,560000,HR Executive,Married,Hatchback
C020,34,690000,Marketing Manager,Single,Sedan
C021,52,1520000,CEO,Married,Luxury Car
C022,37,770000,Teacher,Married,Sedan
C023,25,410000,Graphic Designer,Single,Scooter
C024,40,990000,Data Scientist,Married,SUV
C025,46,1280000,Chartered Accountant,Married,Sedan
C026,32,630000,Pharmacist,Married,Hatchback
C027,29,510000,Civil Engineer,Single,Bike
C028,43,1090000,Professor,Married,SUV
C029,27,450000,Sales Executive,Single,Scooter
C030,35,830000,Software Engineer,Married,Sedan
C031,49,1410000,Business,Married,Luxury Car
C032,38,870000,Lawyer,Married,SUV
C033,24,360000,Student,Single,Bike
C034,31,600000,Nurse,Married,Hatchback
C035,47,1180000,Doctor,Married,SUV
C036,33,680000,Teacher,Single,Scooter
C037,28,540000,Accountant,Married,Hatchback
C038,42,1040000,Engineer,Married,Sedan
C039,36,790000,Data Analyst,Married,SUV
C040,26,430000,Marketing Executive,Single,Bike
C041,53,1650000,Entrepreneur,Married,Luxury Car
C042,39,910000,Government Employee,Married,SUV
C043,34,700000,IT Consultant,Married,Sedan
C044,30,580000,HR Executive,Single,Hatchback
C045,45,1220000,Business,Married,SUV
C046,27,490000,Teacher,Single,Scooter
C047,37,840000,Software Engineer,Married,Sedan
C048,41,970000,Banker,Married,SUV
C049,29,520000,Pharmacist,Single,Hatchback
C050,51,1480000,CEO,Married,Luxury Car"""
    return pd.read_csv(io.StringIO(csv_text))

print("Setup complete. pandas", pd.__version__, "| numpy", np.__version__)

Setup complete. pandas 3.0.2 | numpy 2.4.4


# 02 · Filtering & Categorical Encoding
### From "which rows do I want?" to "how do I turn text into numbers responsibly?"

**Dataset:** `data/customer.csv` — 50 customers of a vehicle dealership, with
`Age`, `Income`, `Profession`, `Marital_Status`, and `Vehicle_Type`. Small and
clean on the surface, but it hides one real trap: `Profession` has **25
distinct values across just 50 rows** — a textbook high-cardinality column
that breaks the "just one-hot encode everything" instinct.

**Learning objectives**
1. **Filtering** — four idiomatic ways to select rows, and when each shines.
2. **Feature extraction via binning** — turn continuous `Age`/`Income` into
   meaningful groups two different ways (`pd.cut` vs `pd.qcut`).
3. **See a real breakage, live** — the original notebook's encoding pattern
   (`inplace=True` replace on a filtered subset) is reproduced verbatim and
   shown failing on today's pandas — then fixed.
4. **Encoding showdown** — six techniques on the same column, compared on
   output shape, assumptions, and when each is the right (or wrong) choice.

**Contents**
1. Load & first look
2. Filtering — four routes to the same kind of answer
3. Categorical vs numeric — and a cardinality check
4. Feature extraction — binning continuous columns
5. The original encoding pattern — and why it now breaks
6. Encoding showdown — six methods, one column
7. A reusable, leakage-safe encoding function

In [2]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)

DATA_PATH = dataset_path('customer.csv', rebuild=rebuild_customer)   # resolved wherever the datasets folder lives

## 1 · Load & first look

In [3]:
df = pd.read_csv(DATA_PATH)
print(f"Shape: {df.shape[0]} rows x {df.shape[1]} columns")
df.head(8)

Shape: 50 rows x 6 columns


,Cust_ID,Age,Income,Profession,Marital_Status,Vehicle_Type
0,C001,24,350000,Student,Single,Bike
1,C002,28,520000,Software Engineer,Married,Hatchback
2,C003,35,780000,Doctor,Married,SUV
3,C004,42,1250000,Business,Married,Sedan
4,C005,31,610000,Teacher,Single,Scooter
5,C006,45,980000,Banker,Married,Sedan
6,C007,29,430000,Sales Executive,Single,Bike
7,C008,38,890000,Professor,Married,SUV


In [4]:
df.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Cust_ID,50,50,C001,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Age,50.0,NaN,NaN,NaN,36.18,8.331622,23.0,29.0,35.5,42.0,53.0
Income,50.0,NaN,NaN,NaN,824200.0,350620.965181,320000.0,525000.0,785000.0,1027500.0,1650000.0
Profession,50,25,Business,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Marital_Status,50,2,Married,34,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Vehicle_Type,50,6,SUV,12,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 2 · Filtering — four routes to the same kind of answer

All four return a filtered `DataFrame`. The right one to reach for depends on
how many conditions you're combining and who else will read the code.

In [5]:
# (1) Boolean mask — the fundamental building block; every other route
#     eventually compiles down to this.
married = df[df["Marital_Status"] == "Married"]

# (2) .loc[] with a mask — identical result, but .loc makes clear you're
#     doing label-based selection, and lets you slice columns in the same call.
over_40 = df.loc[df["Age"] > 40, ["Cust_ID", "Age", "Profession"]]

# (3) .query() — best once you have 2+ conditions; reads like SQL's WHERE
#     clause and avoids repeating `df[...]` for every term.
affluent_married = df.query('Marital_Status == "Married" and Income > 900000')

# (4) .isin() — the natural choice for "is this value one of a set?",
#     instead of chaining several `==` conditions with `|`.
premium_vehicles = df[df["Vehicle_Type"].isin(["SUV", "Luxury Car"])]

print(f"Married:                 {len(married):>3} rows")
print(f"Over 40:                 {len(over_40):>3} rows")
print(f"Married AND income>900k: {len(affluent_married):>3} rows")
print(f"Drives SUV or Luxury:    {len(premium_vehicles):>3} rows")

Married:                  34 rows
Over 40:                  16 rows
Married AND income>900k:  18 rows
Drives SUV or Luxury:     18 rows


In [6]:
# Combining conditions: parentheses are mandatory around each term with & / | —
# unlike plain Python `and`/`or`, these operators are element-wise and bind
# more tightly than comparisons, so `df.A > 1 & df.B < 2` silently does the
# wrong thing without them.
mask = (df["Marital_Status"] == "Single") & (df["Age"] < 30)
young_singles = df[mask]
young_singles

,Cust_ID,Age,Income,Profession,Marital_Status,Vehicle_Type
0,C001,24,350000,Student,Single,Bike
6,C007,29,430000,Sales Executive,Single,Bike
9,C010,27,470000,Accountant,Single,Hatchback
12,C013,26,390000,Nurse,Single,Scooter
15,C016,23,320000,Student,Single,Bike
22,C023,25,410000,Graphic Designer,Single,Scooter
26,C027,29,510000,Civil Engineer,Single,Bike
28,C029,27,450000,Sales Executive,Single,Scooter
32,C033,24,360000,Student,Single,Bike
39,C040,26,430000,Marketing Executive,Single,Bike


## 3 · Categorical vs numeric — and a cardinality check

The original notebook isolated categorical columns with `select_dtypes` and
stopped there. We add one question it didn't ask: **how many distinct values
does each one have?** — because that number decides which encoding strategy
in Section 6 will actually work well.

In [7]:
categorical_cols = df.select_dtypes(exclude=[np.number]).columns.tolist()
categorical_cols.remove("Cust_ID")   # an identifier, not a feature — see callout below
print("Categorical feature columns:", categorical_cols)

Categorical feature columns: ['Profession', 'Marital_Status', 'Vehicle_Type']


In [8]:
for col in categorical_cols:
    print(f"{col:16s} -> {df[col].nunique():2d} unique values "
          f"out of {len(df)} rows")

Profession       -> 25 unique values out of 50 rows
Marital_Status   ->  2 unique values out of 50 rows
Vehicle_Type     ->  6 unique values out of 50 rows


> **Callout — `Cust_ID` is not a feature.** It's a unique row identifier
> (50 values for 50 rows). Encoding it — one-hot or otherwise — would just
> teach a model to memorise row identity, which is meaningless on new
> customers. The fix is simple: exclude ID-like columns before encoding,
> which is why it's dropped from `categorical_cols` above rather than
> encoded "because `select_dtypes` found it."

`Profession` immediately stands out: **25 unique values in 50 rows** — almost
one category per customer. One-hot encoding that as-is would add 25 sparse
columns for a 50-row table. That's the problem Section 6 is built to solve.

## 4 · Feature extraction — binning continuous columns

Binning turns a continuous number into a meaningful group — itself a form of
feature extraction (pulling a coarser, more interpretable signal out of a
precise one). Two standard routes, with different philosophies:

- **`pd.cut`** — *you* define the boundaries (domain knowledge: "under 30 is
  a young adult"). Bin sizes can be uneven.
- **`pd.qcut`** — *the data* defines the boundaries, by splitting into equal-
  sized quantile groups. Useful when you want balanced groups rather than
  human-meaningful thresholds.

In [9]:
# --- pd.cut: fixed, human-defined boundaries -------------------------------
age_bins   = [0, 30, 45, 100]
age_labels = ["Young Adult", "Established", "Senior"]
df["age_group"] = pd.cut(df["Age"], bins=age_bins, labels=age_labels)

df[["Age", "age_group"]].drop_duplicates("age_group").sort_values("Age")

,Age,age_group
0,24,Young Adult
2,35,Established
8,50,Senior


In [10]:
# --- pd.qcut: data-driven quantile boundaries -------------------------------
df["income_tier"] = pd.qcut(df["Income"], q=4, labels=["Low", "Mid", "High", "Premium"])

print(df["income_tier"].value_counts().sort_index(), "\n")
print("Notice: qcut gives ~12-13 customers per tier by construction — cut")
print("would not guarantee that unless you picked boundaries by hand to match.")

income_tier
Low        13
Mid        12
High       12
Premium    13
Name: count, dtype: int64 

Notice: qcut gives ~12-13 customers per tier by construction — cut
would not guarantee that unless you picked boundaries by hand to match.


## 5 · The original encoding pattern — and why it now breaks

The original notebook's approach was:

```python
df_categorical = df.select_dtypes(exclude=[np.number])
df_categorical.Grade.replace({"1st Class": 1, ...}, inplace=True)
```

i.e. take a *subset* of the DataFrame, then mutate a column of that subset
**in place**. We reproduce the exact same pattern below — subset first,
`inplace=True` second — on `Marital_Status`, and check what actually happens
on current pandas.

In [11]:
import warnings

df_categorical = df.select_dtypes(exclude=[np.number])   # a *subset* — the original pattern
before = df_categorical["Marital_Status"].copy()

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    df_categorical.Marital_Status.replace({"Married": 0, "Single": 1}, inplace=True)  # the original pattern

unchanged = df_categorical["Marital_Status"].equals(before)

print(f"Warning raised     : {caught[0].category.__name__ if caught else '(none)'}")
print(f"Data actually changed? -> {'NO' if unchanged else 'yes'}")

Warning raised     : ChainedAssignmentError
Data actually changed? -> NO


**What happened:** pandas raises `ChainedAssignmentError` — which, despite
the name, is a *warning* (visible above your cell, not a crash), because
`ChainedAssignmentError` subclasses `Warning`. That's arguably the more
dangerous outcome, not the safer one: **the code keeps running, gives no
return value to check, and the `replace()` silently did nothing at all** —
confirmed by `unchanged == True` above. In a longer notebook this is exactly
the kind of failure that goes unnoticed for several cells, until something
much later breaks in a confusing way (a model choking on string categories
it expected to already be numeric).

Under the hood: pandas' Copy-on-Write engine (the default and only mode as of
pandas 2.x/3.x) means `df_categorical` behaves like an independent copy the
moment it's sliced out of `df`. Calling `.replace(inplace=True)` on a column
of that copy has nothing durable to write back to — so the warning fires and
the assignment is dropped.

**The fix pandas' own error message recommends** — assign the result back
explicitly instead of relying on `inplace`:

In [12]:
df_categorical = df_categorical.copy()   # be explicit that this is an independent frame
df_categorical["Marital_Status"] = df_categorical["Marital_Status"].replace(
    {"Married": 0, "Single": 1}
)
df_categorical[["Marital_Status"]].head()

,Marital_Status
0,1
1,0
2,0
3,0
4,1


## 6 · Encoding showdown — six methods, one dataset

Same columns, six techniques. Each cell prints what it produces and a
one-line verdict on when to reach for it.

In [13]:
# --- (1) .map() with an explicit dictionary --------------------------------
# Safer than .replace() for this purpose: any value NOT in the dictionary
# becomes NaN (visible, loud) rather than silently staying as unmapped text
# that could later masquerade as a valid numeric-looking category.
df["marital_mapped"] = df["Marital_Status"].map({"Married": 0, "Single": 1})
df[["Marital_Status", "marital_mapped"]].drop_duplicates()

,Marital_Status,marital_mapped
0,Single,1
1,Married,0


In [14]:
# --- (2) Ordered categorical -> integer codes ------------------------------
# The right tool specifically for ORDINAL data — where the categories have a
# genuine rank. income_tier (Low < Mid < High < Premium) qualifies;
# Marital_Status does not (Married isn't "more" than Single).
df["income_tier_code"] = pd.Categorical(
    df["income_tier"], categories=["Low", "Mid", "High", "Premium"], ordered=True
).codes
df[["Income", "income_tier", "income_tier_code"]].drop_duplicates().sort_values("Income")

,Income,income_tier,income_tier_code
15,320000,Low,0
0,350000,Low,0
32,360000,Low,0
12,390000,Low,0
22,410000,Low,0
6,430000,Low,0
28,450000,Low,0
9,470000,Low,0
45,490000,Low,0
26,510000,Low,0


In [15]:
# --- (3) LabelEncoder (sklearn) ---------------------------------------------
# Convenient, but it assigns codes ALPHABETICALLY with no notion of order —
# fine for a nominal column feeding a tree-based model, risky for a linear
# model that would then treat the codes as if they were quantities.
label_encoder = LabelEncoder()
df["vehicle_label"] = label_encoder.fit_transform(df["Vehicle_Type"])
dict(zip(label_encoder.classes_, range(len(label_encoder.classes_))))

{'Bike': 0,
 'Hatchback': 1,
 'Luxury Car': 2,
 'SUV': 3,
 'Scooter': 4,
 'Sedan': 5}

In [16]:
# --- (4a) One-hot via pandas get_dummies ------------------------------------
onehot_pandas = pd.get_dummies(df["Vehicle_Type"], prefix="Vehicle", dtype=int)
onehot_pandas.head()

,Vehicle_Bike,Vehicle_Hatchback,Vehicle_Luxury Car,Vehicle_SUV,Vehicle_Scooter,Vehicle_Sedan
0,1,0,0,0,0,0
1,0,1,0,0,0,0
2,0,0,0,1,0,0
3,0,0,0,0,0,1
4,0,0,0,0,1,0


In [17]:
# --- (4b) One-hot via sklearn OneHotEncoder ---------------------------------
# Prefer this version inside an ML pipeline: it can be fit on TRAIN and
# re-applied to TEST/production data via .transform(), it can gracefully
# ignore categories never seen during training (handle_unknown="ignore"),
# and drop="first" avoids the "dummy variable trap" (perfect multicollinearity
# between the one-hot columns) for models sensitive to it.
ohe = OneHotEncoder(sparse_output=False, drop="first", handle_unknown="ignore")
onehot_sklearn = ohe.fit_transform(df[["Vehicle_Type"]])
onehot_sklearn_df = pd.DataFrame(
    onehot_sklearn, columns=ohe.get_feature_names_out(["Vehicle_Type"]), index=df.index
)
onehot_sklearn_df.head()

,Vehicle_Type_Hatchback,Vehicle_Type_Luxury Car,Vehicle_Type_SUV,Vehicle_Type_Scooter,Vehicle_Type_Sedan
0,0.0,0.0,0.0,0.0,0.0
1,1.0,0.0,0.0,0.0,0.0
2,0.0,0.0,1.0,0.0,0.0
3,0.0,0.0,0.0,0.0,1.0
4,0.0,0.0,0.0,1.0,0.0


In [18]:
# --- (5) Frequency / count encoding — built for high-cardinality columns ---
# Instead of exploding Profession into 25 sparse one-hot columns, represent
# each category by how common it is. One numeric column, no dimensionality
# blow-up, and it still carries real signal (rare professions vs common ones).
profession_freq = df["Profession"].value_counts(normalize=True)
df["profession_freq"] = df["Profession"].map(profession_freq)
df[["Profession", "profession_freq"]].sort_values("profession_freq", ascending=False).head(8)

,Profession,profession_freq
8,Business,0.10
3,Business,0.10
17,Business,0.10
30,Business,0.10
44,Business,0.10
4,Teacher,0.08
21,Teacher,0.08
35,Teacher,0.08


In [19]:
# --- (6) Group rare categories into 'Other', THEN one-hot ------------------
# A middle ground: keep the most common categories identifiable by name,
# collapse the long tail. Concretely shrinks the one-hot footprint below.
TOP_N = 4
top_professions = df["Profession"].value_counts().nlargest(TOP_N).index
df["profession_grouped"] = np.where(
    df["Profession"].isin(top_professions), df["Profession"], "Other"
)

full_onehot    = pd.get_dummies(df["Profession"], prefix="Prof", dtype=int)
grouped_onehot = pd.get_dummies(df["profession_grouped"], prefix="Prof", dtype=int)

print(f"One-hot columns without grouping : {full_onehot.shape[1]}")
print(f"One-hot columns WITH grouping     : {grouped_onehot.shape[1]}")
df["profession_grouped"].value_counts()

One-hot columns without grouping : 25
One-hot columns WITH grouping     : 5


profession_grouped
Other                35
Business              5
Teacher               4
Student               3
Software Engineer     3
Name: count, dtype: int64

| # | Method | Output width | Best for | Watch out for |
|---|--------|-------------|----------|----------------|
| 1 | `.map()` dict | 1 col | Small, known nominal mapping | Unmapped values → NaN (by design — visible) |
| 2 | Ordered categorical codes | 1 col | Genuinely **ordinal** data | Wrong for nominal data — implies an order that isn't real |
| 3 | `LabelEncoder` | 1 col | Tree-based models, nominal data | Linear/distance models may wrongly treat codes as magnitudes |
| 4 | One-hot (pandas / sklearn) | *k* cols | Low/medium-cardinality nominal | Column blow-up on high-cardinality features |
| 5 | Frequency encoding | 1 col | High-cardinality nominal | Two different categories with equal frequency become indistinguishable |
| 6 | Group rare + one-hot | *n*≪*k* cols | High-cardinality, need interpretability | Choice of `TOP_N` is a judgment call — document it |

## 7 · A reusable, leakage-safe encoding function

Turning the winning choices into one function is what actually makes this
*usable* on the next dataset, not just a one-off script. It fits a `sklearn`
encoder on the **training data only** and reuses it for new data — the
correct order of operations to avoid leaking test-set information into how
categories get encoded.

In [20]:
def engineer_customer_features(raw_df: pd.DataFrame, fitted_encoders: dict | None = None):
    '''
    Turn a raw customer.csv-shaped frame into a model-ready one.

    Parameters
    ----------
    raw_df : the raw dataframe (same columns as customer.csv)
    fitted_encoders : pass the dict returned by an earlier call (on TRAINING
        data) to apply the *same* fitted encoders to new data, e.g. a test
        set — this is what keeps the pipeline leakage-safe.

    Returns
    -------
    (engineered_df, fitted_encoders)
    '''
    out = raw_df.copy()

    out["age_group"] = pd.cut(out["Age"], bins=[0, 30, 45, 100],
                               labels=["Young Adult", "Established", "Senior"])
    out["income_tier"] = pd.qcut(out["Income"], q=4,
                                  labels=["Low", "Mid", "High", "Premium"])
    out["marital_encoded"] = out["Marital_Status"].map({"Married": 0, "Single": 1})

    if fitted_encoders is None:
        # First call (training data): fit fresh encoders and remember them.
        vehicle_ohe = OneHotEncoder(sparse_output=False, drop="first", handle_unknown="ignore")
        vehicle_ohe.fit(out[["Vehicle_Type"]])
        profession_freq = out["Profession"].value_counts(normalize=True)
        fitted_encoders = {"vehicle_ohe": vehicle_ohe, "profession_freq": profession_freq}
    else:
        # Subsequent call (e.g. test data): REUSE what was fit on training data.
        vehicle_ohe = fitted_encoders["vehicle_ohe"]
        profession_freq = fitted_encoders["profession_freq"]

    vehicle_cols = vehicle_ohe.get_feature_names_out(["Vehicle_Type"])
    vehicle_encoded = pd.DataFrame(vehicle_ohe.transform(out[["Vehicle_Type"]]),
                                    columns=vehicle_cols, index=out.index)
    out = pd.concat([out, vehicle_encoded], axis=1)

    # map() on unseen-at-fit-time categories correctly yields NaN — visible,
    # not silently wrong; a sensible downstream default is the training mean.
    out["profession_freq"] = out["Profession"].map(profession_freq).fillna(profession_freq.mean())

    return out, fitted_encoders


engineered, encoders = engineer_customer_features(df[["Cust_ID", "Age", "Income",
                                                        "Profession", "Marital_Status",
                                                        "Vehicle_Type"]])
engineered.head()

,Cust_ID,Age,Income,Profession,Marital_Status,Vehicle_Type,age_group,income_tier,marital_encoded,Vehicle_Type_Hatchback,Vehicle_Type_Luxury Car,Vehicle_Type_SUV,Vehicle_Type_Scooter,Vehicle_Type_Sedan,profession_freq
0,C001,24,350000,Student,Single,Bike,Young Adult,Low,1,0.0,0.0,0.0,0.0,0.0,0.06
1,C002,28,520000,Software Engineer,Married,Hatchback,Young Adult,Low,0,1.0,0.0,0.0,0.0,0.0,0.06
2,C003,35,780000,Doctor,Married,SUV,Established,Mid,0,0.0,0.0,1.0,0.0,0.0,0.04
3,C004,42,1250000,Business,Married,Sedan,Established,Premium,0,0.0,0.0,0.0,0.0,1.0,0.10
4,C005,31,610000,Teacher,Single,Scooter,Established,Mid,1,0.0,0.0,0.0,1.0,0.0,0.08


### Recap

- **Filtering**: boolean mask, `.loc`, `.query()`, `.isin()` — same job, four
  tools, pick by how many conditions and who's reading the code next.
- **Extraction**: `pd.cut` (you set the edges) vs `pd.qcut` (the data sets
  the edges) turn continuous columns into meaningful groups.
- **A real breakage**: the original `inplace=True`-on-a-subset pattern now
  raises `ChainedAssignmentError` on current pandas — reproduced, explained,
  and fixed.
- **Six encoders, one verdict table** — matched to nominal vs ordinal data
  and to low vs high cardinality, instead of one-hot-encoding everything by
  default.
- **One function** — `engineer_customer_features()` — that does it all in a
  leakage-safe, reusable way.

**Where this feeds next:** `03_encoding_masterclass.ipynb` scales these same
ideas up to a real missing-data-heavy dataset (Titanic) and a full
train/test-safe pipeline.

### ⚡ Beyond the syllabus — collapse rare categories before you encode

A column with 40 categories where 25 appear fewer than ten times will produce 40 one-hot columns, most of them almost entirely zero. Those columns add width and noise but almost no information, and they break when an unseen category turns up later. Rolling everything below a threshold into `'Other'` is a two-line fix that improves both the model and its stability.

In [21]:
bank = load_data('banking.csv', rebuild=rebuild_banking)
cat_cols = bank.select_dtypes(include='object').columns.tolist()
target_col = max(cat_cols, key=lambda c: bank[c].nunique())
print(f"Highest-cardinality column: '{target_col}' with {bank[target_col].nunique()} categories\n")

counts = bank[target_col].value_counts()
print("Category frequencies (top 8 and bottom 5):")
print(counts.head(8).to_string())
print("  ...")
print(counts.tail(5).to_string())

MIN_COUNT = max(5, int(len(bank) * 0.01))     # 🔧 CHANGE THIS: your rare-category threshold
rare = counts[counts < MIN_COUNT].index
collapsed = bank[target_col].where(~bank[target_col].isin(rare), 'Other')

print(f"\nThreshold: fewer than {MIN_COUNT} occurrences -> 'Other'")
print(f"  categories before : {bank[target_col].nunique()}")
print(f"  categories after  : {collapsed.nunique()}")
print(f"  rows relabelled   : {bank[target_col].isin(rare).sum():,} "
      f"({bank[target_col].isin(rare).mean():.1%} of the data)")

wide  = pd.get_dummies(bank[target_col], prefix=target_col, dtype=int)
narrow = pd.get_dummies(collapsed, prefix=target_col, dtype=int)
print(f"\nOne-hot columns without collapsing: {wide.shape[1]}")
print(f"One-hot columns after collapsing  : {narrow.shape[1]}  "
      f"({(1 - narrow.shape[1]/wide.shape[1]):.0%} narrower)")

print("""
SAY THIS IN YOUR ANSWER
  "Categories occurring fewer than N times were grouped as 'Other'. This affected X% of
   rows and reduced the encoded width from A to B columns, at negligible information cost
   — and it means an unseen category at prediction time maps safely to 'Other' rather
   than causing an error."
""")

Loaded 'banking.csv' from /home/claude/work/build/Python-BDA-Complete-Notes/datasets


Highest-cardinality column: 'job' with 12 categories

Category frequencies (top 8 and bottom 5):
job
admin.           10425
blue-collar       9254
technician        6746
services          3970
management        2924
retired           1721
entrepreneur      1456
self-employed     1421
  ...
job
self-employed    1421
housemaid        1061
unemployed       1014
student           877
unknown           330

Threshold: fewer than 411 occurrences -> 'Other'
  categories before : 12
  categories after  : 12
  rows relabelled   : 330 (0.8% of the data)

One-hot columns without collapsing: 12
One-hot columns after collapsing  : 12  (0% narrower)

SAY THIS IN YOUR ANSWER
  "Categories occurring fewer than N times were grouped as 'Other'. This affected X% of
   rows and reduced the encoded width from A to B columns, at negligible information cost
   — and it means an unseen category at prediction time maps safely to 'Other' rather
   than causing an error."



---

## Exam quick-reference

| To do this | Write this |
|---|---|
| Category frequencies | `s.value_counts()` |
| Rare categories | `vc[vc < 10].index` |
| Collapse to 'Other' | `s.where(~s.isin(rare), 'Other')` |
| Filter rows out | `df = df[df['col'] != 'unknown']` |
| Report what was dropped | `before - after` row counts |
| Encode what remains | `pd.get_dummies(df, drop_first=True)` |

### Adapting this in the exam

- 'Remove records where…' → filter, then report the count removed.
- 'The column has too many categories' → collapse the rare ones, then encode.

### Traps that cost marks

- Filtering rows changes every statistic that follows. Always print the row count before and after.
- Collapsing categories loses information — justify the threshold you chose.
- Filter *before* encoding, or you'll create one-hot columns for categories you then delete.
- Excluding rows because they're inconvenient rather than invalid is not cleaning.